In [ ]:
import os
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    os.chdir('/content/bistro/script')

repo_root = Path(os.path.join('..')).resolve()
src_root = Path(os.path.join('..', 'src')).resolve()
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

import numpy as np
import pandas as pd

from gluonts.dataset.pandas import PandasDataset
from gluonts.dataset.split import split

from uni2ts.model.moirai import MoiraiForecast, MoiraiModule

from inference_util import plot_publication_forecast_comparison, ar1_forecast

from preprocessing_util import (
    aggregate_daily_forecast_to_monthly,
    prepare_long_df_monthly_for_daily_inference,
)

In [ ]:
MODEL_REPO = repo_root / 'bistro-finetuned'

FREQ = 'M'

PDT = 12
CTX = 240
PSZ = 32
BSZ = 32

ROLLING_WINDOWS = 1
WINDOW_DISTANCE = 20

FORECAST_START_DATE = '2023-01-01'

config = {
    "MODEL_REPO": str(MODEL_REPO),
    "PDT": PDT,
    "CTX": CTX,
    "PSZ": PSZ,
    "BSZ": BSZ,
    "ROLLING_WINDOWS": ROLLING_WINDOWS,
    "WINDOW_DISTANCE": WINDOW_DISTANCE,
    "FORECAST_START_DATE": FORECAST_START_DATE,
}

In [ ]:
series_cpi = repo_root / 'data' / 'bis_cpi_us_yoy_m.csv'
series_wti = repo_root / 'data' / 'wti_m.csv'

df_cpi = pd.read_csv(series_cpi, index_col=0)
df_cpi.index = pd.to_datetime(df_cpi.index).to_period(freq=FREQ)

df_wti = pd.read_csv(series_wti, index_col=0)
df_wti.index = pd.to_datetime(df_wti.index).to_period(freq=FREQ)

df_wti.columns = ["wti_price"]

In [ ]:
df_wti["wti_yoy"] = df_wti["wti_price"].pct_change(12) * 100
df_wti = df_wti[["wti_yoy"]]

In [ ]:
df = df_cpi.copy()
df["item_id"] = "cpi_us_yoy_m"

df = df.merge(df_wti, left_index=True, right_index=True, how="inner")

df.columns = ["target", "item_id", "wti_yoy"]

target_col = "target"

In [ ]:
prep = prepare_long_df_monthly_for_daily_inference(
    df,
    item_id_col="item_id",
    target_col=target_col,
    feat_dynamic_real_cols=["wti_yoy"],
    past_dynamic_real_cols=[],
    freq=FREQ,
    forecast_start_date=FORECAST_START_DATE,
    pdt_patches=PDT,
    ctx_patches=CTX,
    steps_per_period=PSZ,
    rolling_windows=ROLLING_WINDOWS,
    window_distance_patches=WINDOW_DISTANCE,
)

if prep.windows < 1:
    raise ValueError(
        f'Not enough test data after cutoff {prep.train_end} to create a window: '
        f'test_len={(prep.df_dt.index > prep.cutoff_date_dt).sum()} periods, PDT={PDT}.'
    )

In [ ]:
ds = PandasDataset.from_long_dataframe(
    prep.daily_long_df,
    item_id="item_id",
    target="target",
    feat_dynamic_real=["wti_yoy"],
    past_feat_dynamic_real=[],
)

train, test_template = split(ds, date=prep.cutoff_period_daily)

test_data = test_template.generate_instances(
    prediction_length=prep.pdt_steps,
    windows=prep.windows,
    distance=prep.dist_steps,
    max_history=prep.ctx_steps,
)

print(
    f"windows={prep.windows}, pdt_steps={prep.pdt_steps}, "
    f"ctx_steps={prep.ctx_steps}, dist_steps={prep.dist_steps}"
)

In [ ]:
model = MoiraiForecast(
    module=MoiraiModule.from_pretrained(str(MODEL_REPO)),
    prediction_length=int(prep.pdt_steps),
    context_length=int(prep.ctx_steps),
    patch_size=int(PSZ),
    num_samples=int(config.get("NUM_SAMPLES", 100)),
    target_dim=1,
    feat_dynamic_real_dim=ds.num_feat_dynamic_real,
    past_feat_dynamic_real_dim=ds.num_past_feat_dynamic_real,
)

predictor = model.create_predictor(batch_size=BSZ)

inputs = list(test_data.input)
labels = list(test_data.label)
forecasts = list(predictor.predict(test_data.input))

In [ ]:
bistro_monthly_by_window = {}
rmse_rows = []

for w in range(prep.windows):
    samples = np.asarray(forecasts[w].samples, dtype=float)
    label_target = np.asarray(labels[w]["target"], dtype=float)

    inp_target = (
        np.asarray(inputs[w]["target"], dtype=float)
        if "target" in inputs[w]
        else np.asarray([], dtype=float)
    )
    last_input = float(inp_target[-1]) if inp_target.size > 0 else None

    preds, _, ci = aggregate_daily_forecast_to_monthly(
        samples,
        label_target,
        last_input,
        steps_per_period=PSZ,
        expected_periods=PDT,
    )

    pred_index = pd.period_range(
        start=prep.forecast_start + w * WINDOW_DISTANCE,
        periods=PDT,
        freq=FREQ,
    )

    dfw = pd.DataFrame(
        {
            "bistro_pred": preds,
            "bistro_lo": ci[:, 0],
            "bistro_hi": ci[:, 1],
        },
        index=pred_index,
    )

    actual = prep.df_monthly_target[target_col].reindex(pred_index).astype(float)
    pred = dfw["bistro_pred"].astype(float)

    valid = actual.notna() & pred.notna()
    rmse_bistro = (
        float(np.sqrt(np.mean((pred[valid] - actual[valid]) ** 2)))
        if valid.any()
        else np.nan
    )

    bistro_monthly_by_window[w] = dfw

    rmse_rows.append(
        {
            "window": w,
            "test_start": pred_index[0],
            "test_end": pred_index[-1],
            "rmse_bistro": rmse_bistro,
            "n_valid": int(valid.sum()),
        }
    )

rmse_table = pd.DataFrame(rmse_rows)
rmse_table

In [ ]:
bistro_monthly_by_window[0]

In [ ]:
w = 0

forecast_start_w = prep.forecast_start + w * WINDOW_DISTANCE
train_end_w = forecast_start_w - 1

df_actual = prep.df_monthly_target[[target_col]].rename(columns={target_col: "actual"})
df_pred = bistro_monthly_by_window[w]

plot_from = forecast_start_w - min(CTX, 120)
plot_to = df_pred.index.max()

df_plot = df_actual.join(df_pred[["bistro_pred"]], how="outer").sort_index()
df_plot = df_plot.loc[plot_from:plot_to]

df_wti_plot = df[["wti_yoy"]].rename(columns={"wti_yoy": "WTI YoY"})
df_plot = df_plot.join(df_wti_plot, how="left")

fig, ax = plot_publication_forecast_comparison(
    df_plot,
    actual_col="actual",
    forecast_cols={
        "bistro_pred": "BISTRO conditional on WTI",
    },
    forecast_start=forecast_start_w,
    title=f"{target_col} — conditional forecast with WTI scenario",
    ylabel="YoY inflation (%)",
    y_fmt="percent",
    percent_scale=100.0,
    savepaths=None,
)

from matplotlib.ticker import PercentFormatter

ax2 = ax.twinx()
x = df_plot.index.to_timestamp()

ax2.plot(
    x,
    df_plot["WTI YoY"].to_numpy(dtype=float),
    lw=1.6,
    label="WTI YoY"
)

ax2.set_ylabel("WTI YoY (%)")
ax2.yaxis.set_major_formatter(PercentFormatter(xmax=100.0))

h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="upper left", frameon=False, ncol=2)

out_dir = repo_root / "script" / "figures"
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "forecast_cpi_conditional_wti.png", bbox_inches="tight")

fig